# Exploración de Datos — CNN+LSTM Biomecánica
**Proyecto:** Reconocimiento de Ejercicios  
**Equipo:** Christopher Zúñiga (C28730) · Adrian Arrieta Orozco (B70734)

In [ ]:
import cv2, mediapipe as mp, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

In [ ]:
# ── Rutas del proyecto (ajusta automáticamente si el notebook se corre desde
#    notebooks/ o desde la raíz del proyecto) ──────────────────────────────────
import os
from pathlib import Path

_cwd     = Path(os.getcwd())
BASE_DIR = _cwd.parent if _cwd.name == "notebooks" else _cwd

DATA_DIR    = BASE_DIR / "data"
MODELS_DIR  = BASE_DIR / "models"
FIGURES_DIR = BASE_DIR / "figures"
DATASET_CSV = DATA_DIR / "pose_landmarks_dataset.csv"
MODEL_PATH  = MODELS_DIR / "pose_landmarker_full.task"
DATASET_DIR = BASE_DIR / "dataset_limpio"

FIGURES_DIR.mkdir(exist_ok=True)
print(f"Raíz del proyecto : {BASE_DIR}")
print(f"Dataset CSV       : {DATASET_CSV}")
print(f"Modelo MediaPipe  : {MODEL_PATH}")


## Extracción parcial — prueba con 1 video

In [ ]:
# Extracción de prueba — 1 video (deadlift_01)
video_path = str(DATASET_DIR / "deadlift" / "deadlift_01.mp4")
video_name = "deadlift_01.mp4"
label      = "deadlift"

cap = cv2.VideoCapture(video_path)
print("Video abierto." if cap.isOpened() else "Error: no se pudo abrir.")
print(f"Frames: {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}  FPS: {cap.get(cv2.CAP_PROP_FPS)}")

BaseOptions           = mp.tasks.BaseOptions
PoseLandmarker        = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode     = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=VisionRunningMode.IMAGE,
)

dataset_rows = []
with PoseLandmarker.create_from_options(options) as landmarker:
    frame_index = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        result    = landmarker.detect(mp_image)
        if result.pose_landmarks:
            landmarks = result.pose_landmarks[0]
            row_data  = {"video_name": video_name, "label": label, "frame_index": frame_index}
            for i, lm in enumerate(landmarks):
                row_data[f"x{i}"] = lm.x; row_data[f"y{i}"] = lm.y
                row_data[f"z{i}"] = lm.z; row_data[f"vis{i}"] = lm.visibility
            dataset_rows.append(row_data)
        frame_index += 1
    cap.release()

df = pd.DataFrame(dataset_rows)
df.to_csv(DATASET_CSV, index=False)
print(f"Extracción completa: {len(df)} frames → {DATASET_CSV}")
df.head()


## Extracción completa — todos los videos del dataset

In [ ]:
from pathlib import Path
import cv2, mediapipe as mp, numpy as np, pandas as pd

CLASSES = [
    (DATASET_DIR / "deadlift",     "deadlift", "correct"),
    (DATASET_DIR / "deadlift_bad", "deadlift", "incorrect"),
    (DATASET_DIR / "pull Up",      "pull_up",  "correct"),
    (DATASET_DIR / "pull_up_bad",  "pull_up",  "incorrect"),
    (DATASET_DIR / "squat",        "squat",    "correct"),
    (DATASET_DIR / "squat_bad",    "squat",    "incorrect"),
]
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".webm"}

BaseOptions           = mp.tasks.BaseOptions
PoseLandmarker        = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode     = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=VisionRunningMode.IMAGE,
)

all_rows = []
with PoseLandmarker.create_from_options(options) as landmarker:
    for folder_path, exercise, form in CLASSES:
        videos = sorted([f for f in folder_path.iterdir()
                         if f.suffix.lower() in VIDEO_EXTENSIONS], key=lambda p: p.name.lower())
        print(f"\n▶ {folder_path.name}  ({len(videos)} videos)  label={form}")
        for vid in videos:
            cap = cv2.VideoCapture(str(vid))
            if not cap.isOpened():
                print(f"  ✗ {vid.name}"); continue
            frame_index = frames_detected = 0
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break
                rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                result = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
                if result.pose_landmarks:
                    lms = result.pose_landmarks[0]
                    row = {"video_name": vid.name, "exercise": exercise,
                           "label": form, "frame_index": frame_index}
                    for i, lm in enumerate(lms):
                        row[f"x{i}"] = lm.x; row[f"y{i}"] = lm.y
                        row[f"z{i}"] = lm.z; row[f"vis{i}"] = lm.visibility
                    all_rows.append(row); frames_detected += 1
                frame_index += 1
            cap.release()
            print(f"  ✓ {vid.name:40s}  {frames_detected}/{frame_index} frames")

df_full = pd.DataFrame(all_rows)
df_full.to_csv(DATASET_CSV, index=False)
print(f"\nDone! {len(df_full)} filas → {DATASET_CSV}")
print(df_full.groupby(["exercise", "label"]).size().to_string())


## Revisar poses faltantes o nulas en dataset

In [ ]:
df = pd.read_csv(DATASET_CSV)
print(df.shape)
print(df.head())
print(df.dtypes)


In [ ]:
df = pd.read_csv(DATASET_CSV)
frames_per_video = df.groupby(["exercise", "label", "video_name"]).size().reset_index(name="frames_detected")

missing = frames_per_video[frames_per_video["frames_detected"] < 30]
print(f"Videos con detecciones incompletas: {len(missing)}")
print(missing)

print("\nFrames totales por (exercise, label):")
print(frames_per_video.groupby(["exercise", "label"])["frames_detected"].sum())


In [9]:
frames_per_video = df.groupby(["exercise", "label", "video_name"]).size().reset_index(name="frames_detected")               

# Distribución de cuántos frames por video                              
print(frames_per_video.groupby(["exercise", "label"])["frames_detected"].describe().round(1))                       

# Los peores: videos con muy pocos frames                               
print("\nVideos con menos de 15 frames:")                 
print(frames_per_video[frames_per_video["frames_detected"] <
15].sort_values("frames_detected"))

                    count  mean  std   min   25%   50%   75%   max
exercise label                                                    
deadlift correct     25.0  29.9  0.4  28.0  30.0  30.0  30.0  30.0
         incorrect   25.0  23.4  9.3   4.0  18.0  28.0  30.0  30.0
pull_up  correct     25.0  26.6  5.3  13.0  26.0  30.0  30.0  30.0
         incorrect   25.0  24.3  7.4   4.0  22.0  27.0  30.0  30.0
squat    correct     25.0  29.3  1.2  26.0  29.0  30.0  30.0  30.0
         incorrect   20.0  30.0  0.0  30.0  30.0  30.0  30.0  30.0

Videos con menos de 15 frames:
    exercise      label           video_name  frames_detected
34  deadlift  incorrect  deadlift_bad_10.mp4                4
85   pull_up  incorrect   pull_up_bad_11.mp4                4
33  deadlift  incorrect  deadlift_bad_09.mp4                7
25  deadlift  incorrect  deadlift_bad_01.mp4                8
43  deadlift  incorrect  deadlift_bad_19.mp4                9
44  deadlift  incorrect  deadlift_bad_20.mp4               1

In [ ]:
# ── Step 4 — Histogramas: distribución de coordenadas clave ──────────────────
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns

df = pd.read_csv(DATASET_CSV)

LANDMARK_POR_EJERCICIO = {
    "deadlift": ("y23", "Cadera izq. (y23)\nBisagra de cadera"),
    "pull_up":  ("y11", "Hombro izq. (y11)\nElevación del cuerpo"),
    "squat":    ("y25", "Rodilla izq. (y25)\nProfundidad de sentadilla"),
}
COLORES = {"correct": "#27ae60", "incorrect": "#c0392b"}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Distribución de coordenada Y por ejercicio y forma de ejecución",
             fontsize=13, fontweight="bold", y=1.02)

for ax, (exercise, (col, xlabel)) in zip(axes, LANDMARK_POR_EJERCICIO.items()):
    subset = df[df["exercise"] == exercise]
    sns.histplot(data=subset, x=col, hue="label", bins=35, kde=True, ax=ax,
                 palette=COLORES, alpha=0.55, hue_order=["correct", "incorrect"])
    ax.set_title(exercise.replace("_", " ").title(), fontsize=12, fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel("Nº de frames")
    ax.legend(title="Forma", labels=["Correcta", "Incorrecta"], fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "hist_landmark_y.png", dpi=150, bbox_inches="tight")
plt.show()

print("Estadísticas descriptivas por ejercicio y forma:")
for exercise, (col, _) in LANDMARK_POR_EJERCICIO.items():
    stats = df[df["exercise"] == exercise].groupby("label")[col].describe().round(3)
    print(f"\n{exercise.upper()} — {col}:")
    print(stats[["mean", "std", "min", "50%", "max"]])


In [ ]:
import cv2, mediapipe as mp
orig_folder = BASE_DIR / "Dataset_Wrong Exercises" / "Dataset_Wrong Exercises" / "Squat_Wrong"
orig_videos = sorted(orig_folder.iterdir(), key=lambda p: p.name)

BaseOptions           = mp.tasks.BaseOptions
PoseLandmarker        = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode     = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=VisionRunningMode.IMAGE,
)

with PoseLandmarker.create_from_options(options) as landmarker:
    for vid in orig_videos:
        cap = cv2.VideoCapture(str(vid))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        detected = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
            if result.pose_landmarks: detected += 1
        cap.release()
        pct = detected / total * 100 if total > 0 else 0
        print(f"{vid.name:10s}  {detected:4d}/{total:4d} frames detectados  ({pct:.0f}%)")


In [ ]:
# ── Step 5 — Heatmap de correlación entre landmarks clave ─────────────────────
KEY_Y   = ["y11", "y23", "y25", "y27"]
NOMBRES = {"y11": "Hombro", "y23": "Cadera", "y25": "Rodilla", "y27": "Tobillo"}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Correlación entre landmarks clave (coord. Y)\npor ejercicio y forma",
             fontsize=13, fontweight="bold")

for col_idx, exercise in enumerate(["deadlift", "pull_up", "squat"]):
    for row_idx, form in enumerate(["correct", "incorrect"]):
        ax     = axes[row_idx, col_idx]
        subset = df[(df["exercise"] == exercise) & (df["label"] == form)][KEY_Y].rename(columns=NOMBRES)
        sns.heatmap(subset.corr(), ax=ax, annot=True, fmt=".2f",
                    cmap="coolwarm", vmin=-1, vmax=1, square=True,
                    linewidths=0.5, cbar=(col_idx == 2))
        ax.set_title(f"{exercise.replace('_',' ').title()}\n"
                     f"({'Correcta' if form=='correct' else 'Incorrecta'})",
                     fontsize=10, fontweight="bold",
                     color="#27ae60" if form == "correct" else "#c0392b")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "heatmap_correlacion.png", dpi=150, bbox_inches="tight")
plt.show()

print("Diferencia en correlación cadera-rodilla (correct - incorrect):")
for exercise in ["deadlift", "pull_up", "squat"]:
    corr_c = df[(df["exercise"]==exercise) & (df["label"]=="correct")][KEY_Y].corr()
    corr_i = df[(df["exercise"]==exercise) & (df["label"]=="incorrect")][KEY_Y].corr()
    diff   = corr_c.loc["y23","y25"] - corr_i.loc["y23","y25"]
    print(f"  {exercise:10s}  correct={corr_c.loc['y23','y25']:.3f}  "
          f"incorrect={corr_i.loc['y23','y25']:.3f}  Δ={diff:+.3f}")


In [ ]:
# ── Step 6 — Box plots: proxies de ángulos articulares ───────────────────────
df["squat_depth_proxy"] = (df["y25"] - df["y23"]).abs()
df["arm_bend_proxy"]    = (df["y13"] - df["y11"]).abs()
df["torso_angle_proxy"] = (df["y11"] - df["y23"]).abs()

PROXIES = {
    "squat":    ("squat_depth_proxy",  "Profundidad sentadilla\n|y_rodilla − y_cadera|"),
    "pull_up":  ("arm_bend_proxy",     "Doblado de brazo\n|y_codo − y_hombro|"),
    "deadlift": ("torso_angle_proxy",  "Inclinación de tronco\n|y_hombro − y_cadera|"),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Proxies de ángulos articulares por ejercicio y forma", fontsize=13, fontweight="bold")

for ax, (exercise, (proxy_col, ylabel)) in zip(axes, PROXIES.items()):
    subset = df[df["exercise"] == exercise]
    sns.boxplot(data=subset, x="label", y=proxy_col, ax=ax,
                palette={"correct": "#27ae60", "incorrect": "#c0392b"},
                order=["correct", "incorrect"], width=0.5, linewidth=1.5,
                flierprops=dict(marker="o", markersize=3, alpha=0.4))
    ax.set_title(exercise.replace("_", " ").title(), fontsize=12, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xticklabels(["Correcta", "Incorrecta"])

plt.tight_layout()
plt.savefig(FIGURES_DIR / "boxplot_proxies.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"{'Ejercicio':<12} {'Proxy':<22} {'Correcta':>10} {'Incorrecta':>12} {'Δ':>8}")
print("-" * 68)
for exercise, (proxy_col, _) in PROXIES.items():
    subset = df[df["exercise"] == exercise]
    med_c  = subset[subset["label"] == "correct"][proxy_col].median()
    med_i  = subset[subset["label"] == "incorrect"][proxy_col].median()
    print(f"{exercise:<12} {proxy_col:<22} {med_c:>10.4f} {med_i:>12.4f} {med_c-med_i:>+8.4f}")
